In [1]:
import numpy as np
import cv2
import torch
import os
import math
import time
from collections import deque
from torchvision import transforms
from utils.datasets import letterbox
from utils.general import non_max_suppression_kpt
from utils.plots import output_to_keypoint, plot_skeleton_kpts
from models.yolo import Model

class FallDetector:
    def __init__(self):
        """
        Initialize the Fall Detector with parameters as defined in the paper
        "Enhanced Fall Detection Using YOLOv7-W6-Pose for Real-Time Elderly Monitoring"
        
        Key parameters:
        - LENGTH_FACTOR_ALPHA (α): Used in height condition formula (Section 3.1)
        - VELOCITY_THRESHOLD: Threshold for fall speed detection (Section 3.2)
        - LEG_ANGLE_THRESHOLD: Degrees threshold for leg angles (Section 3.2)
        - TORSO_ANGLE_THRESHOLD: Degrees threshold for torso orientation (Section 3.2)
        - ASPECT_RATIO_THRESHOLD: Width/height ratio threshold (Section 3.1)
        - CONFIDENCE_THRESHOLD: Minimum keypoint confidence for reliable detection
        """
        # Threshold parameters as defined in the paper
        self.LENGTH_FACTOR_ALPHA = 0.5  # α in the height condition formula
        self.VELOCITY_THRESHOLD = 1.0    # px/frame for fall speed
        self.LEG_ANGLE_THRESHOLD = 45    # degrees for leg angles
        self.TORSO_ANGLE_THRESHOLD = 50  # degrees for torso orientation
        self.ASPECT_RATIO_THRESHOLD = 0.8 # width/height ratio
        self.CONFIDENCE_THRESHOLD = 0.4  # minimum keypoint confidence
        self.TARGET_FPS = 25
        
        # State tracking variables
        self.prev_keypoints = None
        self.velocity_buffer = deque(maxlen=3)  # tracks vertical speed
        self.fall_buffer = deque(maxlen=2)      # confirmation buffer
        self.prev_frame_time = None
        self.fall_start_time = None
        self.prev_shoulder_y = None

    def calculate_euclidean_distance(self, point1, point2):
        """ (Keep this method the same as in your original code) """
        return math.hypot(point1[0]-point2[0], point1[1]-point2[1])

    def calculate_angle(self, a, b, c):
        """ (Keep this method the same as in your original code) """
        try:
            ba = np.array([a[0]-b[0], a[1]-b[1]])
            bc = np.array([c[0]-b[0], c[1]-b[1]])
            cosine_angle = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc) + 1e-6)
            return np.degrees(np.arccos(np.clip(cosine_angle, -1.0, 1.0)))
        except:
            return 180  # return maximum angle if calculation fails

    def calculate_torso_angle(self, shoulders, hips):
        """ (Keep this method the same as in your original code) """
        shoulder_center = np.mean(shoulders, axis=0)
        hip_center = np.mean(hips, axis=0)
        vertical_vector = np.array([0, 1])
        torso_vector = np.array([hip_center[0]-shoulder_center[0], 
                                hip_center[1]-shoulder_center[1]])
        
        if np.linalg.norm(torso_vector) < 1e-6:
            return 90  # neutral angle if points overlap
            
        cosine = np.dot(torso_vector, vertical_vector) / (np.linalg.norm(torso_vector) + 1e-6)
        return np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

    def detect_fall(self, keypoints):
        """
        Main fall detection function implementing the paper's algorithm from Sections 3.1 and 3.2
        Combines multiple conditions (height, velocity, angles, aspect ratio) to detect falls
        
        Args:
            keypoints: Array of 17 keypoints with (x,y,confidence)
        Returns:
            tuple: (is_fall, debug_info) where debug_info contains condition information
        """
        # Keypoint indices as defined in the paper
        NOSE = 0
        LEFT_SHOULDER = 5
        RIGHT_SHOULDER = 6
        LEFT_HIP = 11
        RIGHT_HIP = 12
        LEFT_KNEE = 13
        RIGHT_KNEE = 14
        LEFT_ANKLE = 15
        RIGHT_ANKLE = 16
        
        debug_info = {
            'height_cond': False,
            'speed_cond': False,
            'leg_angle_cond': False,
            'torso_cond': False,
            'aspect_cond': False,
            'values': {},
            'conditions_met': 0
        }
        
        try:
            # Extract keypoints with confidence check
            kp = {
                'nose': keypoints[NOSE*3:(NOSE+1)*3],
                'left_shoulder': keypoints[LEFT_SHOULDER*3:(LEFT_SHOULDER+1)*3],
                'right_shoulder': keypoints[RIGHT_SHOULDER*3:(RIGHT_SHOULDER+1)*3],
                'left_hip': keypoints[LEFT_HIP*3:(LEFT_HIP+1)*3],
                'right_hip': keypoints[RIGHT_HIP*3:(RIGHT_HIP+1)*3],
                'left_knee': keypoints[LEFT_KNEE*3:(LEFT_KNEE+1)*3],
                'right_knee': keypoints[RIGHT_KNEE*3:(RIGHT_KNEE+1)*3],
                'left_ankle': keypoints[LEFT_ANKLE*3:(LEFT_ANKLE+1)*3],
                'right_ankle': keypoints[RIGHT_ANKLE*3:(RIGHT_ANKLE+1)*3]
            }
            
            # Confidence check for all keypoints
            critical_keypoints = ['left_shoulder', 'right_shoulder', 'left_hip', 'right_hip']
            if any(kp[part][2] < self.CONFIDENCE_THRESHOLD for part in critical_keypoints):
                return False, debug_info

            # Get coordinates (convert to tuples for clarity)
            ls = (kp['left_shoulder'][0], kp['left_shoulder'][1])
            rs = (kp['right_shoulder'][0], kp['right_shoulder'][1])
            lh = (kp['left_hip'][0], kp['left_hip'][1])
            rh = (kp['right_hip'][0], kp['right_hip'][1])
            lk = (kp['left_knee'][0], kp['left_knee'][1])
            rk = (kp['right_knee'][0], kp['right_knee'][1])
            la = (kp['left_ankle'][0], kp['left_ankle'][1])
            ra = (kp['right_ankle'][0], kp['right_ankle'][1])

            """ 1. HEIGHT CONDITION (Paper Section 3.1) """
            # Calculate length factor (Lfactor) as Euclidean distance
            torso_mid = ((lh[0] + rh[0])/2, (lh[1] + rh[1])/2)
            Lfactor = self.calculate_euclidean_distance(ls, torso_mid)
            
            # Get vertical positions
            max_feet_y = max(la[1], ra[1]) if kp['left_ankle'][2] > 0.2 and kp['right_ankle'][2] > 0.2 else max(lk[1], rk[1])
            min_shoulder_y = min(ls[1], rs[1])
            
            # Paper's height condition: yl ≤ yFl + α·Lfactor
            height_cond = min_shoulder_y >= (max_feet_y - self.LENGTH_FACTOR_ALPHA * Lfactor)
            debug_info['height_cond'] = height_cond
            debug_info['values']['lfactor'] = Lfactor
            debug_info['values']['height_diff'] = min_shoulder_y - max_feet_y
            
            """ 2. VELOCITY CONDITION (Paper Section 3.2) """
            current_time = time.time()
            vertical_speed = 0
            current_min_y = min(ls[1], rs[1])
            
            if self.prev_shoulder_y is not None and self.prev_frame_time is not None:
                time_elapsed = current_time - self.prev_frame_time
                if time_elapsed > 0:
                    vertical_speed = (current_min_y - self.prev_shoulder_y) / time_elapsed
                    self.velocity_buffer.append(abs(vertical_speed))
            
            avg_speed = sum(self.velocity_buffer)/len(self.velocity_buffer) if self.velocity_buffer else 0
            speed_cond = avg_speed >= self.VELOCITY_THRESHOLD
            debug_info['speed_cond'] = speed_cond
            debug_info['values']['vertical_speed'] = avg_speed
            
            """ 3. ANGLE CONDITIONS (Paper Section 3.2) """
            # Check if knee and ankle keypoints are detected with sufficient confidence
            use_leg_angle = (kp['left_knee'][2] > 0.2 and kp['left_ankle'][2] > 0.2 and 
                             kp['right_knee'][2] > 0.2 and kp['right_ankle'][2] > 0.2)
            
            if use_leg_angle:
                left_leg_angle = self.calculate_angle(lh, lk, la)
                right_leg_angle = self.calculate_angle(rh, rk, ra)
                min_leg_angle = min(left_leg_angle, right_leg_angle)
                leg_angle_cond = min_leg_angle < self.LEG_ANGLE_THRESHOLD
                debug_info['values']['min_leg_angle'] = min_leg_angle
            else:
                leg_angle_cond = False
                debug_info['values']['min_leg_angle'] = -1  # indicates not calculated
            
            debug_info['leg_angle_cond'] = leg_angle_cond
            
            # Torso orientation (mentioned in Section 3.2)
            torso_angle = self.calculate_torso_angle([ls, rs], [lh, rh])
            torso_cond = torso_angle > self.TORSO_ANGLE_THRESHOLD
            debug_info['torso_cond'] = torso_cond
            debug_info['values']['torso_angle'] = torso_angle
            
            """ 4. ASPECT RATIO CONDITION (Paper Section 3.1) """
            # Body orientation ratio: width/height
            body_width = abs(ls[0] - rs[0])
            head_to_feet = abs(kp['nose'][1] - max_feet_y) if kp['nose'][2] > 0.2 else abs(min_shoulder_y - max_feet_y)
            orientation_ratio = body_width / (head_to_feet + 1e-6)
            aspect_cond = orientation_ratio > self.ASPECT_RATIO_THRESHOLD
            debug_info['aspect_cond'] = aspect_cond
            debug_info['values']['orientation_ratio'] = orientation_ratio
            
            """ FALL DECISION LOGIC (Paper Section 3.4.2) """
            # According to the paper, at least 2 conditions must be met
            conditions_met = sum([height_cond, speed_cond, leg_angle_cond, torso_cond, aspect_cond])
            is_fall = conditions_met >= 2
            
            # Add confirmation buffer to reduce false positives
            self.fall_buffer.append(is_fall)
            final_detection = sum(self.fall_buffer) >= 2 if len(self.fall_buffer) >= 1 else is_fall
            
            # Add conditions_met to debug_info
            debug_info['conditions_met'] = conditions_met
            
            # Update tracking variables for next frame
            self.prev_keypoints = kp
            self.prev_shoulder_y = current_min_y
            self.prev_frame_time = current_time
            
            return final_detection, debug_info
            
        except Exception as e:
            print(f"Detection error: {str(e)}")
            return False, debug_info


def process_video_with_output(video_path, model, detector, output_path, annotation_path=None):
    """
    Process a video file to detect falls using YOLOv7-W6-Pose and save output video
    with detection annotations as shown in the journal
    
    Args:
        video_path: Path to the video file
        model: YOLOv7-W6-Pose model instance
        detector: FallDetector instance
        output_path: Path to save the processed video
        annotation_path: Optional path to ground truth annotation file
    Returns:
        List of frames where falls were detected
    """
    # Initialize device
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return []

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    # Create VideoWriter object
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    # Target FPS as mentioned in Section 3.3 (Data Collection and Preprocessing)
    target_fps = detector.TARGET_FPS  # Use the detector's target FPS
    skip_frames = max(1, int(round(fps / target_fps)))  # Fixed the missing parenthesis here
    
    detected_frames = []
    frame_counter = 0
    annotation_range = parse_annotation(annotation_path) if annotation_path else None
    
    # Create FONT settings for visualization
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    font_thickness = 2
    
    # Colors for visualization
    green_color = (0, 255, 0)  # For "Person detected!" text
    blue_color = (255, 0, 0)   # For "No person detected!" text
    red_color = (0, 0, 255)    # For "Person fell down" text
    bbox_color = (0, 255, 255) # For bounding box

    print(f"Processing video with {total_frames} frames, fps={fps}, skip={skip_frames}")

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame_counter += 1
        if frame_counter % skip_frames != 0 and frame_counter < total_frames - 5:
            # Skip frames for processing efficiency, but still write them to output
            out.write(frame)
            continue

        # Create a copy of the frame for drawing
        output_frame = frame.copy()
        
        # Preprocess frame - letterbox resizing as mentioned in Section 3.3
        img = letterbox(frame, 960, stride=64, auto=True)[0]
        img_tensor = transforms.ToTensor()(img)
        img_tensor = torch.tensor(np.array([img_tensor.numpy()]))
        
        if torch.cuda.is_available():
            img_tensor = img_tensor.half().to(device)

        # Inference with YOLOv7-W6-Pose model (Section 3.4)
        with torch.no_grad():
            output, _ = model(img_tensor)
            output = non_max_suppression_kpt(output, 0.25, 0.65, nc=model.yaml['nc'], nkpt=model.yaml['nkpt'], kpt_label=True)
            output = output_to_keypoint(output)

        # Process detections using fall detection algorithm
        if len(output) > 0:
            # Person(s) detected
            cv2.putText(output_frame, "Person detected!", (10, 30), font, font_scale, green_color, font_thickness)
            
            for idx in range(output.shape[0]):
                # Get bounding box
                bbox = output[idx, :4]
                
                # Get keypoints and use them to create a better bounding box
                keypoints = output[idx, 7:].T
                x_coords = []
                y_coords = []
                
                # Extract keypoint coordinates and filter by confidence
                for i in range(17):  # 17 keypoints
                    conf = keypoints[i*3+2]
                    if conf > 0.2:  # Only use keypoints with sufficient confidence
                        x = keypoints[i*3]
                        y = keypoints[i*3+1]
                        x_coords.append(x)
                        y_coords.append(y)
                
                if x_coords and y_coords:  # If we have any valid keypoints
                    # Find the extremes of the keypoints
                    min_x, max_x = min(x_coords), max(x_coords)
                    min_y, max_y = min(y_coords), max(y_coords)
                    
                    # Add padding around the keypoints
                    padding = 20  # pixels in the resized image
                    min_x = max(0, min_x - padding)
                    min_y = max(0, min_y - padding)
                    max_x = max_x + padding
                    max_y = max_y + padding
                    
                    # Scale to original image size
                    scale_x = width / img.shape[1]
                    scale_y = height / img.shape[0]
                    
                    x1 = int(min_x * scale_x)
                    y1 = int(min_y * scale_y)
                    x2 = int(max_x * scale_x)
                    y2 = int(max_y * scale_y)
                    
                    # Ensure within image boundaries
                    x1 = max(0, min(x1, width - 1))
                    y1 = max(0, min(y1, height - 1))
                    x2 = max(0, min(x2, width - 1))
                    y2 = max(0, min(y2, height - 1))
                    
                    # Draw bounding box based on keypoints
                    cv2.rectangle(output_frame, (x1, y1), (x2, y2), bbox_color, 2)
                else:
                    # Fallback to the original bbox if no valid keypoints
                    x1, y1, x2, y2 = bbox
                    
                    # Scale to original image size
                    scale_x = width / img.shape[1]
                    scale_y = height / img.shape[0]
                    
                    x1 = int(x1 * scale_x)
                    y1 = int(y1 * scale_y)
                    x2 = int(x2 * scale_x)
                    y2 = int(y2 * scale_y)
                    
                    # Ensure within image boundaries
                    x1 = max(0, min(x1, width - 1))
                    y1 = max(0, min(y1, height - 1))
                    x2 = max(0, min(x2, width - 1))
                    y2 = max(0, min(y2, height - 1))
                    
                    # Draw the fallback bounding box
                    cv2.rectangle(output_frame, (x1, y1), (x2, y2), bbox_color, 2)
                
                # Detect fall using the improved method
                is_fall, debug_info = detector.detect_fall(keypoints)
                
                # Print debug information
                debug_str = f"Frame {frame_counter}: conditions_met={debug_info['conditions_met']}, "
                debug_str += f"h={debug_info['height_cond']}, s={debug_info['speed_cond']}, "
                debug_str += f"l={debug_info['leg_angle_cond']}, t={debug_info['torso_cond']}, a={debug_info['aspect_cond']}"
                print(debug_str)
                
                if is_fall:
                    # Mark this frame as containing a fall
                    detected_frames.append(frame_counter)
                    
                    # Add "Person fell down" text
                    cv2.putText(output_frame, "Person fell down", (10, 60), font, font_scale, red_color, font_thickness)
                    
                    # Change bounding box color for fall
                    cv2.rectangle(output_frame, (x1, y1), (x2, y2), red_color, 2)
        else:
            # No person detected
            cv2.putText(output_frame, "No person detected!", (10, 30), font, font_scale, blue_color, font_thickness)
        
        # Write the frame to the output video
        out.write(output_frame)
        
        # Progress update every 100 frames
        if frame_counter % 100 == 0:
            print(f"Processed {frame_counter}/{total_frames} frames, detected falls: {len(detected_frames)}")

    cap.release()
    out.release()
    
    print(f"Video processing complete. Output saved to {output_path}")
    print(f"Detected falls at frames: {detected_frames}")
    
    return detected_frames

def draw_keypoints(frame, keypoints, scale_x, scale_y):
    """
    Draw detected keypoints on the frame
    
    Args:
        frame: The frame to draw on
        keypoints: Array of keypoints with (x,y,confidence)
        scale_x: Scale factor for x coordinates
        scale_y: Scale factor for y coordinates
    """
    # Define keypoint connections for visualization
    skeleton = [
        [16, 14], [14, 12], [17, 15], [15, 13], [12, 13], [6, 12], [7, 13],
        [6, 7], [6, 8], [7, 9], [8, 10], [9, 11], [2, 3], [1, 2], [1, 3],
        [2, 4], [3, 5], [4, 6], [5, 7]
    ]
    
    # Define colors for keypoints and connections
    colors = [(0, 255, 255), (0, 255, 0), (255, 0, 0), (0, 0, 255)]
    
    # Draw keypoints
    for i in range(17):
        x = int(keypoints[i*3] * scale_x)
        y = int(keypoints[i*3+1] * scale_y)
        conf = keypoints[i*3+2]
        
        if conf > 0.2:  # Only draw keypoints with reasonable confidence
            cv2.circle(frame, (x, y), 3, colors[i % len(colors)], -1)
    
    # Draw skeleton connections
    for pair in skeleton:
        idx1, idx2 = pair
        
        # Adjust for 0-indexing (skeleton is defined with 1-indexing)
        idx1 -= 1
        idx2 -= 1
        
        # Get coordinates
        x1 = int(keypoints[idx1*3] * scale_x)
        y1 = int(keypoints[idx1*3+1] * scale_y)
        x2 = int(keypoints[idx2*3] * scale_x)
        y2 = int(keypoints[idx2*3+1] * scale_y)
        
        # Get confidences
        conf1 = keypoints[idx1*3+2]
        conf2 = keypoints[idx2*3+2]
        
        # Draw line if both keypoints have sufficient confidence
        if conf1 > 0.2 and conf2 > 0.2:
            cv2.line(frame, (x1, y1), (x2, y2), colors[(idx1+idx2) % len(colors)], 2)

def parse_annotation(annotation_path):
    """
    Parse ground truth annotation file for fall frames
    Used to evaluate the system against the Le2i dataset as described in the paper (Section 3.6)
    
    Args:
        annotation_path: Path to annotation file
    Returns:
        Tuple of (start_frame, end_frame) for the fall event or None if no fall
    """
    try:
        with open(annotation_path, 'r') as f:
            lines = f.readlines()
            if len(lines) >= 2:
                start_frame = lines[0].strip()
                end_frame = lines[1].strip()
                
                # Handle cases where annotation indicates no fall
                if start_frame in ['0', '00'] and end_frame in ['0', '00']:
                    return None
                
                return (int(start_frame), int(end_frame))
    except Exception as e:
        print(f"Error reading annotation: {str(e)}")
        return None

def evaluate_fall_detection(detected_frames, gt_range):
    """
    Evaluate fall detection results against ground truth
    Implements the evaluation methodology described in the paper (Section 3.6)
    
    Args:
        detected_frames: List of frames where falls were detected
        gt_range: Tuple of (start_frame, end_frame) or None if no fall
        
    Returns:
        bool: True if detection matches ground truth, False otherwise
    """
    # Case 1: No fall in ground truth (gt_range is None)
    if gt_range is None:
        return len(detected_frames) == 0  # Correct if no detections
    
    # Case 2: Fall in ground truth (gt_range is (start, end))
    gt_start, gt_end = gt_range
    for frame in detected_frames:
        if gt_start <= frame <= gt_end:
            return True  # Correct detection
    return False  # Missed detection

def test_scenario_with_output(dataset_path, scenario_name, model, detector, overall_metrics, output_dir):
    """
    Test fall detection on a specific scenario from the dataset and save output videos
    
    Args:
        dataset_path: Path to the dataset root directory
        scenario_name: Name of the scenario to test (e.g., "Coffee_room_01")
        model: Pre-loaded YOLOv7-W6-Pose model
        detector: FallDetector instance
        overall_metrics: Dictionary to accumulate metrics across scenarios
        output_dir: Directory to save output videos
    """
    metrics = {
        'true_positives': 0,
        'false_positives': 0,
        'false_negatives': 0,
        'true_negatives': 0,
        'total_videos': 0
    }
    
    # Find annotation and video folders
    scenario_path = os.path.join(dataset_path, scenario_name)
    
    # Handle different naming conventions in the dataset
    annotation_dirs = ["Annotation_files", "Annotations_files"]
    annotation_dir = None
    for dir_name in annotation_dirs:
        potential_dir = os.path.join(scenario_path, dir_name)
        if os.path.exists(potential_dir):
            annotation_dir = potential_dir
            break
    
    videos_dir = os.path.join(scenario_path, "Videos")
    
    # Create output directory if it doesn't exist
    scenario_output_dir = os.path.join(output_dir, scenario_name.replace(os.path.sep, "_"))
    os.makedirs(scenario_output_dir, exist_ok=True)
    
    if not annotation_dir or not os.path.exists(videos_dir):
        print(f"Could not find required folders in {scenario_path}")
        return
    
    print(f"\nTesting scenario: {scenario_name}")
    
    # Process each video in the scenario
    for video_file in os.listdir(videos_dir):
        if not video_file.lower().endswith(('.avi', '.mp4', '.mov')):
            continue
            
        video_name = os.path.splitext(video_file)[0]
        video_path = os.path.join(videos_dir, video_file)
        annotation_file = os.path.join(annotation_dir, f"{video_name}.txt")
        output_path = os.path.join(scenario_output_dir, f"{video_name}_output.mp4")
        
        if not os.path.exists(annotation_file):
            print(f"Annotation not found for {video_file}")
            continue
            
        print(f"\nProcessing video: {video_file}")
        detected_frames = process_video_with_output(video_path, model, detector, output_path, annotation_file)
        
        # Parse ground truth
        gt_range = parse_annotation(annotation_file)  # This returns None if no fall
        
        # Evaluate prediction
        if gt_range is None:
            # Case: No fall in ground truth
            metrics['total_videos'] += 1
            if detected_frames:
                metrics['false_positives'] += 1
                print("Result: WRONG (False alarm - detected fall when there was none)")
            else:
                metrics['true_negatives'] += 1
                print("Result: CORRECT (No fall detected as expected)")
        else:
            # Case: Fall in ground truth
            metrics['total_videos'] += 1
            is_correct = evaluate_fall_detection(detected_frames, gt_range)
            
            if is_correct:
                metrics['true_positives'] += 1
                print("Result: CORRECT (Fall detected within ground truth range)")
            else:
                if detected_frames:
                    metrics['false_positives'] += 1
                    print("Result: WRONG (Fall detected outside ground truth range)")
                else:
                    metrics['false_negatives'] += 1
                    print("Result: WRONG (No fall detected)")
        
        print(f"Ground truth fall frames: {gt_range if gt_range is not None else 'No fall'}")
        print(f"Detected fall frames: {detected_frames}")
    
    # Calculate and print scenario metrics
    if metrics['total_videos'] > 0:
        TP = metrics['true_positives']
        FP = metrics['false_positives']
        FN = metrics['false_negatives']
        TN = metrics['true_negatives']
        
        # Calculate evaluation metrics as described in the paper (Section 4.1)
        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        accuracy = (TP + TN) / metrics['total_videos'] if metrics['total_videos'] > 0 else 0
        
        print(f"\nScenario Results for {scenario_name}:")
        print(f"Total Videos Processed: {metrics['total_videos']}")
        print(f"True Positives: {TP}")
        print(f"False Positives: {FP}")
        print(f"False Negatives: {FN}")
        print(f"True Negatives: {TN}")
        print(f"Precision: {precision:.2%}")
        print(f"Recall: {recall:.2%}")
        print(f"F1-Score: {f1:.2%}")
        print(f"Accuracy: {accuracy:.2%}")
        
        # Update overall metrics
        for key in metrics:
            overall_metrics[key] += metrics[key]
    else:
        print(f"\nNo valid videos processed for {scenario_name}")

def main():
    """
    Main function to run fall detection on all datasets with output videos
    """
    # Set dataset path
    dataset_path = r"F:\PROJECTS\Maching Learning & Artificial Intelligence\yolov7-w6pose-replicate\datasets"
    
    # Set output directory for processed videos
    output_dir = r"F:\PROJECTS\Maching Learning & Artificial Intelligence\yolov7-w6pose-replicate\outputs"
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize device and model
    print("Initializing YOLOv7-W6-Pose model...")
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    try:
        weights = torch.load('yolov7-w6-pose.pt', map_location=device, weights_only=False)
        model = weights['model'].float().eval()
        if torch.cuda.is_available():
            model.half().to(device)
        
        detector = FallDetector()
        
        # Initialize overall metrics
        overall_metrics = {
            'true_positives': 0,
            'false_positives': 0,
            'false_negatives': 0,
            'true_negatives': 0,
            'total_videos': 0
        }
        
        # Get all Le2i scenarios
        scenarios = get_le2i_scenarios(dataset_path)
        
        if not scenarios:
            print("No valid scenarios found in Le2i dataset")
        else:
            print(f"Found {len(scenarios)} scenarios: {scenarios}")
            
            # Process each Le2i scenario
            for scenario in scenarios:
                test_scenario_with_output(
                    os.path.join(dataset_path, "le2i"), 
                    scenario, 
                    model, 
                    detector, 
                    overall_metrics,
                    output_dir
                )
            
            # Print overall metrics for Le2i dataset
            if overall_metrics['total_videos'] > 0:
                TP = overall_metrics['true_positives']
                FP = overall_metrics['false_positives']
                FN = overall_metrics['false_negatives']
                TN = overall_metrics['true_negatives']
                
                precision = TP / (TP + FP) if (TP + FP) > 0 else 0
                recall = TP / (TP + FN) if (TP + FN) > 0 else 0
                f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
                accuracy = (TP + TN) / overall_metrics['total_videos'] if overall_metrics['total_videos'] > 0 else 0
                
                print("\n=====================================================")
                print("OVERALL RESULTS ACROSS ALL LE2I SCENARIOS")
                print("=====================================================")
                print(f"Total Videos Processed: {overall_metrics['total_videos']}")
                print(f"True Positives: {TP}")
                print(f"False Positives: {FP}")
                print(f"False Negatives: {FN}")
                print(f"True Negatives: {TN}")
                print(f"Precision: {precision:.2%}")
                print(f"Recall: {recall:.2%}")
                print(f"F1-Score: {f1:.2%}")
                print(f"Accuracy: {accuracy:.2%}")
                
                # Save metrics to a file
                metrics_path = os.path.join(output_dir, "metrics.txt")
                with open(metrics_path, 'w') as f:
                    f.write("OVERALL RESULTS ACROSS ALL LE2I SCENARIOS\n")
                    f.write(f"Total Videos Processed: {overall_metrics['total_videos']}\n")
                    f.write(f"True Positives: {TP}\n")
                    f.write(f"False Positives: {FP}\n")
                    f.write(f"False Negatives: {FN}\n")
                    f.write(f"True Negatives: {TN}\n")
                    f.write(f"Precision: {precision:.2%}\n")
                    f.write(f"Recall: {recall:.2%}\n")
                    f.write(f"F1-Score: {f1:.2%}\n")
                    f.write(f"Accuracy: {accuracy:.2%}\n")
            
    except Exception as e:
        print(f"Error in main execution: {str(e)}")
        import traceback
        traceback.print_exc()

def get_le2i_scenarios(dataset_path):
    """
    Find all Le2i dataset scenarios in the given path
    
    Args:
        dataset_path: Path to the Le2i dataset root directory
    
    Returns:
        List of scenario paths
    """
    scenarios = []
    le2i_path = os.path.join(dataset_path, "le2i")
    
    # Check if path exists and print for debugging
    print(f"Looking for Le2i dataset at: {le2i_path}")
    if not os.path.exists(le2i_path):
        print(f"Error: Le2i dataset path not found at {le2i_path}")
        return scenarios
    
    # Check if we have the updated directory structure with Le2i_Sorted
    le2i_sorted_path = os.path.join(le2i_path, "Le2i_Sorted")
    print(f"Checking for Le2i_Sorted at: {le2i_sorted_path}")
    
    if os.path.exists(le2i_sorted_path):
        print(f"Found Le2i_Sorted directory structure")
        
        # Process both Fall and Non Fall directories
        for category in ["Fall", "Non Fall"]:
            category_path = os.path.join(le2i_sorted_path, category)
            print(f"Checking category: {category} at {category_path}")
            
            if not os.path.exists(category_path):
                print(f"Category path not found: {category_path}")
                continue
                
            # Iterate through scenario directories (Coffee_room_01, Home_01, etc.)
            for scenario_dir in os.listdir(category_path):
                scenario_path = os.path.join(category_path, scenario_dir)
                print(f"Checking scenario: {scenario_dir} at {scenario_path}")
                
                # Skip if not a directory
                if not os.path.isdir(scenario_path):
                    print(f"Not a directory: {scenario_path}")
                    continue
                
                # Check for Videos folder and one of the annotation folder variants
                videos_dir = os.path.join(scenario_path, "Videos")
                annotation_found = False
                
                for ann_dir in ["Annotation_files", "Annotations_files"]:
                    ann_path = os.path.join(scenario_path, ann_dir)
                    if os.path.exists(ann_path):
                        annotation_found = True
                        break
                
                if os.path.exists(videos_dir) and annotation_found:
                    scenario_rel_path = os.path.join("Le2i_Sorted", category, scenario_dir)
                    print(f"Adding valid scenario: {scenario_rel_path}")
                    scenarios.append(scenario_rel_path)
    else:
        print(f"Le2i_Sorted directory not found, checking original structure")
        # Original directory structure (no Le2i_Sorted)
        # Look for scenarios directly under le2i_path
        for item in os.listdir(le2i_path):
            item_path = os.path.join(le2i_path, item)
            if os.path.isdir(item_path):
                videos_dir = os.path.join(item_path, "Videos")
                annotation_found = False
                
                for ann_dir in ["Annotation_files", "Annotations_files"]:
                    ann_path = os.path.join(item_path, ann_dir)
                    if os.path.exists(ann_path):
                        annotation_found = True
                        break
                
                if os.path.exists(videos_dir) and annotation_found:
                    scenarios.append(item)
    
    print(f"Total scenarios found: {len(scenarios)}")
    return scenarios

def process_single_video(video_path, output_path, model, detector):
    """
    Process a single video file to detect falls and save the output
    
    Args:
        video_path: Path to the video file
        output_path: Path to save the output video
        model: YOLOv7-W6-Pose model instance
        detector: FallDetector instance
    """
    detected_frames = process_video_with_output(video_path, model, detector, output_path)
    print(f"Processed {video_path}")
    print(f"Detected falls at frames: {detected_frames}")
    print(f"Output saved to: {output_path}")

if __name__ == "__main__":
    main()

Initializing YOLOv7-W6-Pose model...
Using device: cuda:0
Looking for Le2i dataset at: F:\PROJECTS\Maching Learning & Artificial Intelligence\yolov7-w6pose-replicate\datasets\le2i
Checking for Le2i_Sorted at: F:\PROJECTS\Maching Learning & Artificial Intelligence\yolov7-w6pose-replicate\datasets\le2i\Le2i_Sorted
Found Le2i_Sorted directory structure
Checking category: Fall at F:\PROJECTS\Maching Learning & Artificial Intelligence\yolov7-w6pose-replicate\datasets\le2i\Le2i_Sorted\Fall
Checking scenario: Coffee_room_01 at F:\PROJECTS\Maching Learning & Artificial Intelligence\yolov7-w6pose-replicate\datasets\le2i\Le2i_Sorted\Fall\Coffee_room_01
Adding valid scenario: Le2i_Sorted\Fall\Coffee_room_01
Checking scenario: Coffee_room_02 at F:\PROJECTS\Maching Learning & Artificial Intelligence\yolov7-w6pose-replicate\datasets\le2i\Le2i_Sorted\Fall\Coffee_room_02
Adding valid scenario: Le2i_Sorted\Fall\Coffee_room_02
Checking scenario: Home_01 at F:\PROJECTS\Maching Learning & Artificial Intel

C:\Users\naufa\anaconda3\envs\cv_env\Lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


Frame 9: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 10: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 11: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 12: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 13: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 14: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 15: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 16: conditions_met=0, h=False, s=False, l=False, t=False, a=False
Frame 17: conditions_met=1, h=False, s=True, l=False, t=False, a=False
Frame 18: conditions_met=1, h=False, s=True, l=False, t=False, a=False
Frame 19: conditions_met=1, h=False, s=True, l=False, t=False, a=False
Frame 20: conditions_met=1, h=False, s=True, l=False, t=False, a=False
Frame 21: conditions_met=1, h=False, s=True, l=False, t=False, a=False
Frame 22: conditions_met=1, h=False, s=True, l=False, t=False, a=False

KeyboardInterrupt: 